# Experiment 5: Subword Tokenization and POS Tagging

## 5.1 BPE Tokenization

In [4]:
# a. Using pretrained model (GPT-2 uses BPE)
import os

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

from transformers import GPT2Tokenizer
tokenizer_bpe_pre = GPT2Tokenizer.from_pretrained('gpt2')

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    sample_text = f.read().strip().split('\n')[0] # Using first line as sample

print('--- BPE Pretrained ---')
encoded_bpe = tokenizer_bpe_pre.encode(sample_text)
tokens_bpe = tokenizer_bpe_pre.convert_ids_to_tokens(encoded_bpe)
for t, i in zip(tokens_bpe[:20], encoded_bpe[:20]):
    print(f'{t}: {i}')

--- BPE Pretrained ---
Natural: 35364
Ġlanguage: 3303
Ġprocessing: 7587
Ġis: 318
Ġa: 257
Ġbranch: 8478
Ġof: 286
Ġartificial: 11666
Ġintelligence: 4430
Ġthat: 326


In [5]:
# b. Without pretrained model (Train from scratch)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer_bpe_scratch = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe_scratch.pre_tokenizer = Whitespace()  # pyright: ignore[reportAttributeAccessIssue]
trainer = BpeTrainer(vocab_size=1000, special_tokens=['[UNK]'])
tokenizer_bpe_scratch.train(['input_sub_word_data.txt'], trainer)

print('\n--- BPE From Scratch ---')
encoded_scratch = tokenizer_bpe_scratch.encode(sample_text)
for t, i in zip(encoded_scratch.tokens[:20], encoded_scratch.ids[:20]):
    print(f'{t}: {i}')


--- BPE From Scratch ---
Natural: 355
language: 138
processing: 188
is: 78
a: 20
branch: 874
of: 85
artificial: 920
intelligence: 918
that: 262


## 5.2 SentencePiece Tokenization

In [6]:
# a. Using pretrained model (T5 uses SentencePiece)
from transformers import T5Tokenizer
tokenizer_sp_pre = T5Tokenizer.from_pretrained('t5-small', legacy=False)

print('\n--- SentencePiece Pretrained ---')
encoded_sp = tokenizer_sp_pre.encode(sample_text)
tokens_sp = tokenizer_sp_pre.convert_ids_to_tokens(encoded_sp)
for t, i in zip(tokens_sp[:20], encoded_sp[:20]):
    print(f'{t}: {i}')


--- SentencePiece Pretrained ---
▁Natural: 6869
▁language: 1612
▁processing: 3026
▁is: 19
▁: 3
a: 9
▁branch: 6421
▁of: 13
▁artificial: 7353
▁intelligence: 6123
▁that: 24
</s>: 1


In [12]:
# b. Without pretrained model (Train from scratch)
import sentencepiece as spm  # pyright: ignore[reportMissingImports]
spm.SentencePieceTrainer.train(input='input_sub_word_data.txt', model_prefix='m_sp', vocab_size=400)
sp = spm.SentencePieceProcessor(model_file='m_sp.model')

print('\n--- SentencePiece From Scratch ---')
encoded_sp_scratch = sp.encode_as_ids(sample_text)
tokens_sp_scratch = sp.encode_as_pieces(sample_text)
for t, i in zip(tokens_sp_scratch[:20], encoded_sp_scratch[:20]):
    print(f'{t}: {i}')


--- SentencePiece From Scratch ---
▁Natural: 181
▁language: 19
▁processing: 41
▁is: 16
▁a: 8
▁b: 299
ran: 380
ch: 145
▁of: 11
▁arti: 289
fic: 133
i: 37
al: 103
▁in: 32
te: 99
ll: 298
i: 37
g: 398
ence: 124
▁that: 82


## 5.3 POS Tagging (Short Sentence)

In [10]:
sentence = 'The young student is reading an interesting book in the library.'

# Universal POS descriptions avoid relying on spaCy's private explain export.
POS_DESCRIPTIONS = {
    'ADJ': 'adjective',
    'ADP': 'adposition',
    'ADV': 'adverb',
    'AUX': 'auxiliary verb',
    'CCONJ': 'coordinating conjunction',
    'DET': 'determiner',
    'INTJ': 'interjection',
    'NOUN': 'noun',
    'NUM': ' numeral',
    'PART': 'particle',
    'PRON': 'pronoun',
    'PROPN': 'proper noun',
    'PUNCT': 'punctuation',
    'SCONJ': 'subordinating conjunction',
    'SYM': 'symbol',
    'VERB': 'verb',
    'X': 'other',
}

# a. Using spaCy
import spacy
nlp = spacy.load('en_core_web_sm')
doc = nlp(sentence)

print('\n--- POS Tagging (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token in doc:
    print(f'{token.text:<15} | {token.pos_:<10} | {POS_DESCRIPTIONS.get(token.pos_, token.pos_)}')

# b. Using NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

tokens_nltk = nltk.word_tokenize(sentence)
pos_tags_nltk = nltk.pos_tag(tokens_nltk)

print('\n--- POS Tagging (NLTK) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token, tag in pos_tags_nltk:
    print(f'{token:<15} | {tag:<10} | {tag}')


--- POS Tagging (spaCy) ---
Token           | POS Tag    | Description
--------------------------------------------------
The             | DET        | determiner
young           | ADJ        | adjective
student         | NOUN       | noun
is              | AUX        | auxiliary verb
reading         | VERB       | verb
an              | DET        | determiner
interesting     | ADJ        | adjective
book            | NOUN       | noun
in              | ADP        | adposition
the             | DET        | determiner
library         | NOUN       | noun
.               | PUNCT      | punctuation

--- POS Tagging (NLTK) ---
Token           | POS Tag    | Description
--------------------------------------------------
The             | DT         | DT
young           | JJ         | JJ
student         | NN         | NN
is              | VBZ        | VBZ
reading         | VBG        | VBG
an              | DT         | DT
interesting     | JJ         | JJ
book            | NN         | N

## 5.4 POS Tagging with Frequency (Large File)

In [11]:
from collections import Counter

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    large_text = f.read()

# Process first 5000 chars for demonstration
doc_large = nlp(large_text[:5000])

# Count frequencies of (Token, POS)
pos_freq = Counter([(token.text, token.pos_, POS_DESCRIPTIONS.get(token.pos_, token.pos_)) for token in doc_large if not token.is_space])

print('\n--- POS Tagging with Frequency (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Frequency":<10} | {"Description"}')
print('-'*65)
for (token, pos, desc), freq in pos_freq.most_common(20):
    print(f'{token:<15} | {pos:<10} | {freq:<10} | {desc}')


--- POS Tagging with Frequency (spaCy) ---
Token           | POS Tag    | Frequency  | Description
-----------------------------------------------------------------
,               | PUNCT      | 52         | punctuation
.               | PUNCT      | 52         | punctuation
the             | DET        | 26         | determiner
and             | CCONJ      | 21         | coordinating conjunction
a               | DET        | 19         | determiner
can             | AUX        | 15         | auxiliary verb
of              | ADP        | 13         | adposition
is              | AUX        | 11         | auxiliary verb
language        | NOUN       | 10         | noun
be              | AUX        | 10         | auxiliary verb
-               | PUNCT      | 10         | punctuation
as              | ADP        | 10         | adposition
words           | NOUN       | 9          | noun
word            | NOUN       | 9          | noun
BPE             | PROPN      | 9          | proper no